Connected to Python 3.14.2

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Plotly graphs
pio.templates["group1"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial, sans-serif", size=13),
        title=dict(x=0.02, font=dict(size=18)),
        colorway=px.colors.qualitative.Set2,
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "plotly_white+group1"

Path("figures").mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save a PNG copy to figures/ if possible; otherwise skip quietly."""
    try:
        fig.write_image(f"figures/{name}.png", scale=2)
    except Exception:
        pass

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Plotly graphs
pio.templates["group1"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial, sans-serif", size=13),
        title=dict(x=0.02, font=dict(size=18)),
        colorway=px.colors.qualitative.Set2,
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "plotly_white+group1"

Path("figures").mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save a PNG copy to figures/ if possible; otherwise skip quietly."""
    try:
        fig.write_image(f"figures/{name}.png", scale=2)
    except Exception:
        pass

In [ ]:
ratings = pd.DataFrame(
    {
        "Fuad":     [2, 1, 2, 1, 4, 2, 4, 2, 3],
        "Jonathan": [2, 2, 1, 1, 4, 2, 3, 2, 4],
        "Katie":    [3, 3, 3, 2, 4, 2, 4, 3, 3],
    },
    index=[
        "Python", "SQL", "Machine Learning", "Cloud Computing", "Communication",
        "AWS", "Dashboard/Reporting", "Git", "Financial Domain Knowledge",
    ],
)
df_skills = ratings.T   # rows = members, columns = skills
ratings

,Fuad,Jonathan,Katie
Python,2,2,3
SQL,1,2,3
Machine Learning,2,1,3
Cloud Computing,1,1,2
Communication,4,4,4
AWS,2,2,2
Dashboard/Reporting,4,3,4
Git,2,2,3
Financial Domain Knowledge,3,4,3


In [ ]:
PANEL_PATH = Path("data/processed/career_market_panel.csv")
SKILL_FIELD = "SKILLS_NAME"

# Each team skill area -> regex over cleaned, lower-case skill names
AREA_PATTERNS = {
    "Python":                    r"\bpython\b",
    "SQL":                       r"\bsql\b|mysql|postgresql",
    "Machine Learning":          r"machine learning|deep learning|predictive model|scikit|tensorflow|pytorch",
    "Cloud Computing":           r"cloud|\bazure\b|snowflake|databricks",
    "Communication":             r"communication|presentation|public speaking|storytelling",
    "AWS":                       r"\baws\b|amazon web services|amazon redshift|amazon s3",
    "Dashboard/Reporting":       r"dashboard|\breporting\b|tableau|power bi|looker|qlik|business intelligence|data visuali",
    "Git":                       r"\bgit\b|github|gitlab|version control",
    "Financial Domain Knowledge": r"financ|investment|securities|portfolio|accounting|asset management|risk management|trading|fixed income|derivative|valuation",
}
assert set(AREA_PATTERNS) == set(df_skills.columns), "Area names must match the ratings table"

panel = pd.read_csv(PANEL_PATH, usecols=[SKILL_FIELD], dtype=str)
n_postings = len(panel)

def clean_skill(name):
    # "SQL (Programming Language)" -> "sql"
    return re.sub(r"\s*\([^)]*\)", "", name).strip().casefold()

skills_per_posting = (
    panel[SKILL_FIELD].fillna("").str.split(";")
    .apply(lambda items: {clean_skill(x) for x in items if x.strip()})
)

# Match each distinct skill name once, then map postings to areas
skill_freq = Counter(s for skills in skills_per_posting for s in skills)
skill_to_areas = {
    s: {a for a, p in AREA_PATTERNS.items() if re.search(p, s)} for s in skill_freq
}
areas_per_posting = skills_per_posting.apply(
    lambda skills: set().union(*(skill_to_areas[s] for s in skills))
)

demand = pd.DataFrame({
    "Postings": {a: int(areas_per_posting.apply(lambda x: a in x).sum()) for a in AREA_PATTERNS}
})
demand["Share"] = demand["Postings"] / n_postings * 100
demand = demand.sort_values("Share", ascending=False)
demand.style.format({"Share": "{:.1f}%", "Postings": "{:,}"})

,Postings,Share
Financial Domain Knowledge,"4,720",66.7%
Communication,"1,662",23.5%
Dashboard/Reporting,"1,580",22.3%
SQL,"1,303",18.4%
Python,863,12.2%
Cloud Computing,258,3.6%
Machine Learning,200,2.8%
Git,66,0.9%
AWS,32,0.5%


In [ ]:
PANEL_PATH = Path("data/processed/career_market_panel.csv")
SKILL_FIELD = "SKILLS_NAME"

# Each team skill area -> regex over cleaned, lower-case skill names
AREA_PATTERNS = {
    "Python":                    r"\bpython\b",
    "SQL":                       r"\bsql\b|mysql|postgresql",
    "Machine Learning":          r"machine learning|deep learning|predictive model|scikit|tensorflow|pytorch",
    "Cloud Computing":           r"cloud|\bazure\b|snowflake|databricks",
    "Communication":             r"communication|presentation|public speaking|storytelling",
    "AWS":                       r"\baws\b|amazon web services|amazon redshift|amazon s3",
    "Dashboard/Reporting":       r"dashboard|\breporting\b|tableau|power bi|looker|qlik|business intelligence|data visuali",
    "Git":                       r"\bgit\b|github|gitlab|version control",
    "Financial Domain Knowledge": r"financ|investment|securities|portfolio|accounting|asset management|risk management|trading|fixed income|derivative|valuation",
}
assert set(AREA_PATTERNS) == set(df_skills.columns), "Area names must match the ratings table"

panel = pd.read_csv(PANEL_PATH, usecols=[SKILL_FIELD], dtype=str)
n_postings = len(panel)

def clean_skill(name):
    # "SQL (Programming Language)" -> "sql"
    return re.sub(r"\s*\([^)]*\)", "", name).strip().casefold()

skills_per_posting = (
    panel[SKILL_FIELD].fillna("").str.split(";")
    .apply(lambda items: {clean_skill(x) for x in items if x.strip()})
)

# Match each distinct skill name once, then map postings to areas
skill_freq = Counter(s for skills in skills_per_posting for s in skills)
skill_to_areas = {
    s: {a for a, p in AREA_PATTERNS.items() if re.search(p, s)} for s in skill_freq
}
areas_per_posting = skills_per_posting.apply(
    lambda skills: set().union(*(skill_to_areas[s] for s in skills))
)

demand = pd.DataFrame({
    "Postings": {a: int(areas_per_posting.apply(lambda x: a in x).sum()) for a in AREA_PATTERNS}
})
demand["Share"] = demand["Postings"] / n_postings * 100
demand = demand.sort_values("Share", ascending=False)
demand.style.format({"Share": "{:.1f}%", "Postings": "{:,}"})

,Postings,Share
Financial Domain Knowledge,"4,720",66.7%
Communication,"1,662",23.5%
Dashboard/Reporting,"1,580",22.3%
SQL,"1,303",18.4%
Python,863,12.2%
Cloud Computing,258,3.6%
Machine Learning,200,2.8%
Git,66,0.9%
AWS,32,0.5%


In [ ]:
fig = px.bar(
    demand.reset_index(names="Skill area").sort_values("Share"),
    x="Share", y="Skill area", orientation="h",
    title=f"Share of {n_postings:,} NAICS 523 Analyst Postings Requiring Each Skill Area",
    labels=dict(Share="% of postings", **{"Skill area": ""}),
    hover_data={"Postings": ":,"},
)
save_fig(fig, "sga_market_demand")
fig.show()

In [ ]:
pd.DataFrame({
    "Most frequent matched skills": {
        a: ", ".join(
            sorted((s for s, areas in skill_to_areas.items() if a in areas),
                   key=lambda s: -skill_freq[s])[:6]
        )
        for a in demand.index
    }
})

,Most frequent matched skills
Financial Domain Knowledge,"[""treasury management"", ""merchant services"", ""..."
Communication,"[""business acumen"", ""business problems"", ""data..."
Dashboard/Reporting,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
SQL,"[""microsoft excel"", ""sql"", ""treasury managemen..."
Python,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
Cloud Computing,"[""amazon web services"", ""microsoft excel"", ""pr..."
Machine Learning,"[""python"", ""sql"", ""microsoft excel"", ""language..."
Git,"[""microsoft office"", ""sap ariba"", ""microsoft e..."
AWS,"[""amazon web services"", ""microsoft excel"", ""pr..."


In [ ]:
pd.DataFrame({
    "Most frequent matched skills": {
        a: ", ".join(
            sorted((s for s, areas in skill_to_areas.items() if a in areas),
                   key=lambda s: -skill_freq[s])[:6]
        )
        for a in demand.index
    }
})

,Most frequent matched skills
Financial Domain Knowledge,"[""treasury management"", ""merchant services"", ""..."
Communication,"[""business acumen"", ""business problems"", ""data..."
Dashboard/Reporting,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
SQL,"[""microsoft excel"", ""sql"", ""treasury managemen..."
Python,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
Cloud Computing,"[""amazon web services"", ""microsoft excel"", ""pr..."
Machine Learning,"[""python"", ""sql"", ""microsoft excel"", ""language..."
Git,"[""microsoft office"", ""sap ariba"", ""microsoft e..."
AWS,"[""amazon web services"", ""microsoft excel"", ""pr..."


In [ ]:
unrated = [(s, c) for s, c in skill_freq.most_common() if not skill_to_areas[s]][:15]
(pd.DataFrame(unrated, columns=["Skill (as listed, lower-cased)", "Postings"])
   .assign(Share=lambda d: d["Postings"] / n_postings * 100)
   .style.format({"Share": "{:.1f}%", "Postings": "{:,}"}))

,"Skill (as listed, lower-cased)",Postings,Share
0,[],196,2.8%
1,"[""new product development"", ""business results"", ""continuous development"", ""market research"", ""pricing strategies"", ""market data"", ""swot analysis"", ""underwriting guidelines"", ""business development"", ""product support""]",15,0.2%
2,"[""leadership development"", ""asset protection"", ""loss prevention"", ""first aid"", ""law enforcement"", ""sales"", ""streamlining"", ""operations""]",10,0.1%
3,"[""microsoft word"", ""microsoft excel"", ""bank secrecy act"", ""usa patriot act"", ""due diligence"", ""lifecycle management"", ""case management"", ""transactional analysis"", ""regulatory requirements"", ""banking products""]",9,0.1%
4,"[""waste management"", ""administrative support"", ""office equipment"", ""operations"", ""collections"", ""coordinating"", ""procurement"", ""scheduling""]",7,0.1%
5,"[""program management"", ""organizational skills"", ""continuous monitoring"", ""organizational leadership"", ""regulatory compliance"", ""business administration"", ""project management"", ""social justice"", ""coordinating"", ""automation""]",7,0.1%
6,"[""business success"", ""operations"", ""research""]",7,0.1%
7,"[""informed consent"", ""clinical research"", ""supervision""]",6,0.1%
8,"[""marketing analytics"", ""data lakes"", ""revenue forecasting"", ""critical thinking"", ""customer marketing"", ""detail oriented"", ""business writing"", ""project management"", ""advertising management"", ""management consulting""]",6,0.1%
9,"[""microsoft office"", ""microsoft excel"", ""administrative support"", ""organizational skills"", ""medical research"", ""project coordination"", ""english language"", ""customer service"", ""time management"", ""operations""]",6,0.1%


In [ ]:
target = pd.cut(
    demand["Share"], bins=[-0.1, 5, 15, 30, 50, 100], labels=[1, 2, 3, 4, 5]
).astype(int)

order = list(demand.index)                     # most -> least demanded
df_skills = df_skills[order]
gap = (target - df_skills).clip(lower=0)

fig = px.imshow(
    df_skills,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Blues",
    title="Team Skill Levels (1 = beginner, 5 = expert)",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Level"),
)
save_fig(fig, "sga_team_matrix")
fig.show()

fig = px.imshow(
    gap,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Reds",
    title="Skill Gap: Levels Below Market Target",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Gap"),
)
save_fig(fig, "sga_gap_matrix")
fig.show()

In [ ]:
pd.DataFrame({
    "Most frequent matched skills": {
        a: ", ".join(
            sorted((s for s, areas in skill_to_areas.items() if a in areas),
                   key=lambda s: -skill_freq[s])[:6]
        )
        for a in demand.index
    }
})

,Most frequent matched skills
Financial Domain Knowledge,"[""treasury management"", ""merchant services"", ""..."
Communication,"[""business acumen"", ""business problems"", ""data..."
Dashboard/Reporting,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
SQL,"[""microsoft excel"", ""sql"", ""treasury managemen..."
Python,"[""sql"", ""tableau"", ""python"", ""technical subjec..."
Cloud Computing,"[""amazon web services"", ""microsoft excel"", ""pr..."
Machine Learning,"[""python"", ""sql"", ""microsoft excel"", ""language..."
Git,"[""microsoft office"", ""sap ariba"", ""microsoft e..."
AWS,"[""amazon web services"", ""microsoft excel"", ""pr..."


In [ ]:
target = pd.cut(
    demand["Share"], bins=[-0.1, 5, 15, 30, 50, 100], labels=[1, 2, 3, 4, 5]
).astype(int)

order = list(demand.index)                     # most -> least demanded
df_skills = df_skills[order]
gap = (target - df_skills).clip(lower=0)

fig = px.imshow(
    df_skills,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Blues",
    title="Team Skill Levels (1 = beginner, 5 = expert)",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Level"),
)
save_fig(fig, "sga_team_matrix")
fig.show()

fig = px.imshow(
    gap,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Reds",
    title="Skill Gap: Levels Below Market Target",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Gap"),
)
save_fig(fig, "sga_gap_matrix")
fig.show()

In [ ]:
target = pd.cut(
    demand["Share"], bins=[-0.1, 5, 15, 30, 50, 100], labels=[1, 2, 3, 4, 5]
).astype(int)

order = list(demand.index)                     # most -> least demanded
df_skills = df_skills[order]
gap = (target - df_skills).clip(lower=0)

fig = px.imshow(
    df_skills,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Blues",
    title="Team Skill Levels (1 = beginner, 5 = expert)",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Level"),
)
save_fig(fig, "sga_team_matrix")
fig.show()

fig = px.imshow(
    gap,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Reds",
    title="Skill Gap: Levels Below Market Target",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Gap"),
)
save_fig(fig, "sga_gap_matrix")
fig.show()

In [ ]:
team_view = pd.DataFrame({
    "Market target": target,
    "Team average": df_skills.mean(),
    "Team best": df_skills.max(),
}).loc[order]

fig = px.bar(
    team_view.reset_index(names="Skill area"),
    x="Skill area", y=["Market target", "Team average", "Team best"],
    barmode="group",
    title="Team Proficiency vs. Market Target",
    labels=dict(value="Proficiency (1–5)", variable="", **{"Skill area": ""}),
)
fig.update_yaxes(range=[0, 5.5])
save_fig(fig, "sga_team_vs_market")
fig.show()

uncovered = team_view.index[team_view["Team best"] < team_view["Market target"]].tolist()
print("No team member meets the target for:", ", ".join(uncovered) or "none")

No team member meets the target for: Financial Domain Knowledge


In [ ]:
priority = gap.mul(demand["Share"] / 100, axis=1)

rows = []
for member, scores in priority.iterrows():
    top3 = scores[scores > 0].sort_values(ascending=False).head(3)
    rows.append({
        "Member": member,
        "Learning order (levels to gain)": " → ".join(
            f"{s} (+{gap.loc[member, s]})" for s in top3.index
        ) or "Meets every target",
    })
pd.DataFrame(rows).set_index("Member")

,Learning order (levels to gain)
Member,
Fuad,Financial Domain Knowledge (+2) → SQL (+2)
Jonathan,Financial Domain Knowledge (+1) → SQL (+1)
Katie,Financial Domain Knowledge (+2)


In [ ]:
priority = gap.mul(demand["Share"] / 100, axis=1)

rows = []
for member, scores in priority.iterrows():
    top3 = scores[scores > 0].sort_values(ascending=False).head(3)
    rows.append({
        "Member": member,
        "Learning order (levels to gain)": " → ".join(
            f"{s} (+{gap.loc[member, s]})" for s in top3.index
        ) or "Meets every target",
    })
pd.DataFrame(rows).set_index("Member")

,Learning order (levels to gain)
Member,
Fuad,Financial Domain Knowledge (+2) → SQL (+2)
Jonathan,Financial Domain Knowledge (+1) → SQL (+1)
Katie,Financial Domain Knowledge (+2)


In [ ]:
team_view = pd.DataFrame({
    "Market target": target,
    "Team average": df_skills.mean(),
    "Team best": df_skills.max(),
}).loc[order]

fig = px.bar(
    team_view.reset_index(names="Skill area"),
    x="Skill area", y=["Market target", "Team average", "Team best"],
    barmode="group",
    title="Team Proficiency vs. Market Target",
    labels=dict(value="Proficiency (1–5)", variable="", **{"Skill area": ""}),
)
fig.update_yaxes(range=[0, 5.5])
save_fig(fig, "sga_team_vs_market")
fig.show()

uncovered = team_view.index[team_view["Team best"] < team_view["Market target"]].tolist()
print("No team member meets the target for:", ", ".join(uncovered) or "none")

No team member meets the target for: Financial Domain Knowledge


In [ ]:
target = pd.cut(
    demand["Share"], bins=[-0.1, 5, 15, 30, 50, 100], labels=[1, 2, 3, 4, 5]
).astype(int)

order = list(demand.index)                     # most -> least demanded
df_skills = df_skills[order]
gap = (target - df_skills).clip(lower=0)

fig = px.imshow(
    df_skills,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Blues",
    title="Team Skill Levels (1 = beginner, 5 = expert)",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Level"),
)
save_fig(fig, "sga_team_matrix")
fig.show()

fig = px.imshow(
    gap,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Reds",
    title="Skill Gap: Levels Below Market Target",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Gap"),
)
save_fig(fig, "sga_gap_matrix")
fig.show()

In [ ]:
target = pd.cut(
    demand["Share"], bins=[-0.1, 5, 15, 30, 50, 100], labels=[1, 2, 3, 4, 5]
).astype(int)

order = list(demand.index)                     # most -> least demanded
df_skills = df_skills[order]
gap = (target - df_skills).clip(lower=0)

fig = px.imshow(
    df_skills,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Blues",
    title="Team Skill Levels Heat Map (1 = beginner, 5 = expert)",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Level"),
)
save_fig(fig, "sga_team_matrix")
fig.show()

fig = px.imshow(
    gap,
    text_auto=True, zmin=0, zmax=5, aspect="auto",
    color_continuous_scale="Reds",
    title="Skill Gap: Levels Below Market Target",
    labels=dict(x="Skill area (most → least in demand)", y="", color="Gap"),
)
save_fig(fig, "sga_gap_matrix")
fig.show()